# Pre-M0.2 — NumPy IQ Arrays, Axes, and dtypes

## Unit Objective

Master NumPy ndarray fundamentals through the lens of IQ signal data. You will learn how shape, ndim, axes, and dtypes relate to canonical IQ layout, and gain the ability to reason about array dimensions before executing code.

## What You Should Learn

After this notebook you should be able to:

1. Explain what `shape`, `ndim`, and `size` mean for any NumPy array.
2. Identify which axis corresponds to examples, which to I/Q, and which to time samples in a 3-D IQ tensor.
3. Choose between `float32`, `float64`, `complex64`, and `complex128` and explain the trade-offs.
4. Use `astype`, `.real`, `.imag`, and complex construction `I + 1j*Q` confidently.
5. Build IQ arrays progressively from 1-D vectors to the canonical 3-D layout `X.shape == (N, 2, L)`.

## IQ Data Contract

All notebooks in this lab follow a single canonical layout:

| Axis | Name | Meaning |
|------|------|----------|
| 0 | examples | One entry per recorded or simulated IQ frame |
| 1 | I/Q | 0 = In-phase (I), 1 = Quadrature (Q) |
| 2 | time | Sample index within a frame |

- **Canonical shape**: `X.shape == (N, 2, L)` where `N` = number of examples and `L` = number of time samples.
- **dtype**: `np.float32` unless otherwise stated.
- **Reproducibility**: `SEED = 42`.

In [ ]:
import numpy as np
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

---

## 1 — ndarray, shape, ndim, size

A NumPy **ndarray** is a multidimensional, homogeneous array of fixed-size items. Three attributes describe its geometry:

- **`shape`**: tuple giving the length along each axis.
- **`ndim`**: number of axes (length of `shape`).
- **`size`**: total number of elements (product of all dimensions).

In [ ]:
a = np.array([1.0, 2.0, 3.0], dtype=np.float32)

print(f"a = {a}")
print(f"shape : {a.shape}")
print(f"ndim  : {a.ndim}")
print(f"size  : {a.size}")
print(f"dtype : {a.dtype}")

**Observation**: a 1-D array has `shape == (3,)`, `ndim == 1`, `size == 3`.

### 1.1 — From 1-D to 2-D

A single IQ sample frame (one example) has In-phase and Quadrature components as two separate 1-D vectors. We can stack them into a 2-D array:

In [ ]:
L = 8  # time samples per frame

I = rng.standard_normal(L).astype(np.float32)
Q = rng.standard_normal(L).astype(np.float32)

sample = np.stack([I, Q], axis=0)  # shape (2, L)

print(f"I.shape == {I.shape}")
print(f"Q.shape == {Q.shape}")
print(f"sample.shape == {sample.shape}")
print(f"sample.ndim == {sample.ndim}")

### 1.2 — From 2-D to 3-D: the canonical tensor

When we collect `N` example frames, we get the canonical 3-D layout:

```
X.shape == (N, 2, L)
```

Axis 0 → examples, axis 1 → I/Q, axis 2 → time.

In [ ]:
N = 5

I_all = rng.standard_normal((N, L)).astype(np.float32)
Q_all = rng.standard_normal((N, L)).astype(np.float32)

X = np.stack([I_all, Q_all], axis=1)  # (N, 2, L)

print(f"X.shape == {X.shape}")
print(f"X.ndim  == {X.ndim}")
print(f"X.size  == {X.size}")
print(f"X.dtype == {X.dtype}")

**Checkpoint**: verify that `X.shape == (N, 2, L)` holds.

In [ ]:
assert X.shape == (N, 2, L), f"Expected ({N}, 2, {L}), got {X.shape}"
print("Shape assertion passed.")

---

## 2 — Dtypes for IQ Data

IQ samples are real-valued voltages. The natural dtype is floating point.

| dtype | Bits | Precision (digits) | Use |
|-------|------|-------------------|-----|
| `float32` | 32 | ~7 | Default for IQ storage — good balance |
| `float64` | 64 | ~15 | Intermediates where precision matters |
| `complex64` | 64 (2 × float32) | ~7 | Complex representation of IQ |
| `complex128` | 128 (2 × float64) | ~15 | High-precision complex intermediates |

**Rule of thumb**: store IQ as `float32`; compute with `float64` or complex types when higher precision is needed; cast back for storage.

In [ ]:
x32 = np.array([1.0, 2.0, 3.0], dtype=np.float32)
x64 = x32.astype(np.float64)

print(f"float32: {x32.dtype}, value = {x32}")
print(f"float64: {x64.dtype}, value = {x64}")

### 2.1 — astype: safe casting and copy behaviour

`array.astype(new_dtype)` always returns a **copy** (even when `new_dtype == array.dtype`).

In [ ]:
X_f32 = X.astype(np.float32)   # already float32, but copy
X_f64 = X.astype(np.float64)   # upcast — copy

print(f"X_f32.dtype = {X_f32.dtype}, same object? {X_f32 is X}")
print(f"X_f64.dtype = {X_f64.dtype}")

---

## 3 — Complex Representation of IQ

Many DSP algorithms expect complex-valued signals. The mapping is:

```
z = I + 1j * Q
```

Going back:

```
I = z.real
Q = z.imag
```

These operations preserve information — no data is lost.

In [ ]:
z = I + 1j * Q

print(f"z.dtype  = {z.dtype}")
print(f"z.shape  = {z.shape}")
print(f"z.real   = {z.real[:3]}")
print(f"z.imag   = {z.imag[:3]}")
print(f"np.array_equal(z.real, I): {np.array_equal(z.real, I)}")
print(f"np.array_equal(z.imag, Q): {np.array_equal(z.imag, Q)}")

### 3.1 — Complex from the 3-D tensor

Extract I and Q from `X` and build a complex array of shape `(N, L)`.

In [ ]:
I_from_X = X[:, 0, :]  # shape (N, L)
Q_from_X = X[:, 1, :]  # shape (N, L)

Z = I_from_X + 1j * Q_from_X  # shape (N, L), complex128

print(f"Z.shape == {Z.shape}")
print(f"Z.dtype == {Z.dtype}")

---

## 4 — Reasoning About Axes

Understanding *which axis does what* is the single most important skill when working with multi-dimensional arrays.

### 4.1 — Axis semantics for IQ data

| Axis | Index in `X.shape` | Name | Content |
|------|-------------------|------|---------|
| 0 | `X.shape[0]` | examples | Each slice `X[i]` is one complete IQ frame |
| 1 | `X.shape[1]` | I/Q | `X[:, 0, :]` = all In-phase; `X[:, 1, :]` = all Quadrature |
| 2 | `X.shape[2]` | time | Each slice `X[:, :, k]` is a single time-step across all frames |

### 4.2 — Slicing examples

In [ ]:
# First example frame: shape (2, L)
first_frame = X[0]
print(f"X[0].shape == {first_frame.shape}")

# All In-phase values: shape (N, L)
all_I = X[:, 0, :]
print(f"X[:, 0, :].shape == {all_I.shape}")

# A single time-step across all examples: shape (N, 2)
one_timestep = X[:, :, 0]
print(f"X[:, :, 0].shape == {one_timestep.shape}")

### 4.3 — np.mean along axes

The `axis` parameter of `np.mean` specifies which dimension to collapse.

- `np.mean(X, axis=0)` → average across examples → shape `(2, L)`.
- `np.mean(X, axis=1)` → average across I/Q → shape `(N, L)` — *not meaningful* for IQ data, but illustrative.
- `np.mean(X, axis=2)` → average across time → shape `(N, 2)` — per-frame DC offset.

In [ ]:
mean_across_examples = np.mean(X, axis=0)
print(f"mean across axis 0: shape = {mean_across_examples.shape}")

mean_across_time = np.mean(X, axis=2)
print(f"mean across axis 2: shape = {mean_across_time.shape}")

---

## 5 — Summary Table

Fill in the meaning of each axis as you encounter objects of different shapes.

| Object | Shape | ndim | dtype | Axis 0 | Axis 1 | Axis 2 |
|--------|-------|------|-------|--------|--------|--------|
| `I` | `(L,)` | 1 | float32 | time samples | — | — |
| `Q` | `(L,)` | 1 | float32 | time samples | — | — |
| `sample` | `(2, L)` | 2 | float32 | 0=I, 1=Q | time samples | — |
| `X` | `(N, 2, L)` | 3 | float32 | examples | 0=I, 1=Q | time samples |
| `Z` | `(N, L)` | 2 | complex128 | examples | time samples | — |

---

## 6 — Student Exercises

For each cell below, **write the answer before running**. The point is to build mental models of NumPy array shapes.

### Exercise 6.1 — PREDICT THE SHAPE BEFORE RUNNING

In [ ]:
# What is the shape of this array?
arr = np.zeros((4, 3, 2))

# YOUR ANSWER (write it, then run the cell): arr.shape == ?
print(f"arr.shape == {arr.shape}")

### Exercise 6.2 — PREDICT THE SHAPE BEFORE RUNNING

In [ ]:
# What is the shape after slicing?
data = np.ones((6, 2, 10))
subset = data[1:4, :, 0]

# YOUR ANSWER: subset.shape == ?
print(f"subset.shape == {subset.shape}")

### Exercise 6.3 — PREDICT THE SHAPE BEFORE RUNNING

In [ ]:
# What shape results from this operation?
I_vec = np.array([1.0, 2.0, 3.0, 4.0])  # shape (4,)
Q_vec = np.array([0.1, 0.2, 0.3, 0.4])  # shape (4,)
stitched = np.stack([I_vec, Q_vec], axis=0)

# YOUR ANSWER: stitched.shape == ?
print(f"stitched.shape == {stitched.shape}")

### Exercise 6.4 — PREDICT THE SHAPE BEFORE RUNNING

In [ ]:
# What shape results from this np.mean call?
X_ex = np.ones((8, 2, 20))
result = np.mean(X_ex, axis=1)

# YOUR ANSWER: result.shape == ?
print(f"result.shape == {result.shape}")

### Exercise 6.5 — PREDICT THE SHAPE BEFORE RUNNING

In [ ]:
# What happens to shape when you convert to complex?
I_3d = rng.standard_normal((4, 10)).astype(np.float32)
Q_3d = rng.standard_normal((4, 10)).astype(np.float32)
Z_3d = I_3d + 1j * Q_3d

# YOUR ANSWER: Z_3d.shape == ?  Z_3d.dtype == ?
print(f"Z_3d.shape == {Z_3d.shape}")
print(f"Z_3d.dtype == {Z_3d.dtype}")

---

## 7 — Pass Criterion Challenges

You must pass all three gates (PC-1, PC-2, PC-3) to proceed.

### PC-1 — Independent Axis Explanation

In the cell below, answer **all six questions** in Markdown. Do NOT run any code — this is a conceptual check.

1. In `X.shape == (N, 2, L)`, what does axis 0 represent?
2. What does axis 1 represent? Which index gives In-phase, which gives Quadrature?
3. What does axis 2 represent?
4. What is the shape of `X[0]`? Explain why.
5. What is the shape of `X[:, 0, :]`? Explain why.
6. If you run `np.mean(X, axis=2)`, what shape results and what does each element represent?

#### STUDENT AXIS EXPLANATION

*(Write your answers to questions 1–6 here before continuing.)*

---

In [ ]:
# STUDENT: change this to True after your instructor verifies your explanation.
AXES_EXPLANATION_VERIFIED = False  # Do NOT auto-evaluate — manual check required

---

### PC-2 — Injected Axis Swap

A well-meaning colleague has transposed the axes of `X`. The resulting array `X_swapped` has shape `(N, L, 2)` instead of `(N, 2, L)`. This breaks every downstream operation that expects the canonical layout.

**Your task:**
1. Inspect `X_swapped.shape` and explain why the I/Q axis is now in the wrong position.
2. Write the correction using `np.transpose` or equivalent.
3. Store the result in `X_fixed` and verify it matches the original `X`.

In [ ]:
# --- Injected error: I/Q axis moved to the wrong position ---
X_swapped = np.transpose(X, (0, 2, 1))  # shape (N, L, 2) — WRONG

print(f"X_swapped.shape == {X_swapped.shape}  ← should be ({N}, 2, {L})")

#### STUDENT ATTEMPT

Fill in the cell below. What is the correct transpose permutation to restore `X`?

In [ ]:
# YOUR CODE HERE
# X_fixed = ...  (restore from X_swapped)

# Uncomment and complete:
# X_fixed = np.transpose(X_swapped, (???))  # replace ??? with correct permutation
# print(f"X_fixed.shape == {X_fixed.shape}")

---

#### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

The swap moved axis 2 (time) to position 1 and axis 1 (I/Q) to position 2.
To undo it, reverse the permutation: `(0, 2, 1)`.

In [ ]:
X_fixed = np.transpose(X_swapped, (0, 2, 1))
print(f"X_fixed.shape == {X_fixed.shape}")

In [ ]:
axis_swap_corrected = (
    X_fixed.shape == X.shape
    and X_fixed.dtype == X.dtype
    and np.array_equal(X_fixed, X)
)
assert X_fixed.shape == X.shape
assert X_fixed.dtype == X.dtype
assert np.array_equal(X_fixed, X)

print("AXIS SWAP CHECK: PASS" if axis_swap_corrected else "AXIS SWAP CHECK: WAIT")

---

### PC-3 — IQ Power Agreement

Signal power can be computed in two completely independent ways:

- **Method A** (real arithmetic): $P = \frac{1}{L}\sum_{k}(I_k^2 + Q_k^2)$
- **Method B** (complex arithmetic): $P = \frac{1}{L}\sum_{k}|z_k|^2$ where $z = I + jQ$

Your task: implement **both methods independently** (neither calls the other) and verify they agree within floating-point tolerance.

#### STUDENT ATTEMPT

Complete both functions below. Use only the formulas given above — do not call one from inside the other.

In [ ]:
def power_method_a(I, Q):
    """Method A: real arithmetic. Return scalar power."""
    # YOUR CODE HERE
    # P_iq = ...
    pass

def power_method_b(I, Q):
    """Method B: complex arithmetic. Return scalar power."""
    # YOUR CODE HERE
    # P_complex = ...
    pass

---

#### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

Method A: `np.mean(I**2 + Q**2)` — pure real arithmetic.

Method B: build `z = I + 1j*Q`, then `np.mean(np.abs(z)**2)`.

In [ ]:
def power_method_a(I, Q):
    """Method A: real arithmetic. Return scalar power."""
    return np.mean(I**2 + Q**2)

def power_method_b(I, Q):
    """Method B: complex arithmetic. Return scalar power."""
    z = I + 1j * Q
    return np.mean(np.abs(z)**2)

In [ ]:
# Compute power using both methods for the first example frame
I_frame = X[0, 0, :]  # shape (L,)
Q_frame = X[0, 1, :]  # shape (L,)

P_iq = power_method_a(I_frame, Q_frame)
P_complex = power_method_b(I_frame, Q_frame)

print(f"Method A (real):      P = {P_iq:.10f}")
print(f"Method B (complex):   P = {P_complex:.10f}")

In [ ]:
POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

power_consistency = np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
assert power_consistency, f"Power mismatch: {P_iq} vs {P_complex}"

print(f"P_iq      = {P_iq:.10f}")
print(f"P_complex = {P_complex:.10f}")
print(f"Absolute diff = {abs(P_iq - P_complex):.2e}")
print(f"POWER CHECK: PASS")

---

## 8 — Automatic Validations

In [ ]:
# Verify canonical shape still holds
assert X.shape == (N, 2, L), f"Canonical shape violated: {X.shape}"
assert X.dtype == np.float32, f"Wrong dtype: {X.dtype}"
print("Canonical shape and dtype: OK")

# Verify axis swap correction
print(f"axis_swap_corrected = {axis_swap_corrected}")

# Verify power consistency
print(f"power_consistency = {power_consistency}")

---

## 9 — Manual Evaluation of Axis Explanation

PC-1 requires human verification. The notebook cannot auto-evaluate your written explanation.

Your instructor (or you, after self-review) must set `AXES_EXPLANATION_VERIFIED = True` in the PC-1 cell above.

---

## 10 — PASS CRITERION GATE

In [ ]:
automatic_checks = {
    "axis_swap_corrected": axis_swap_corrected,
    "power_consistency": power_consistency,
}
automatic_pass = all(automatic_checks.values())
final_pass = automatic_pass and AXES_EXPLANATION_VERIFIED

print("=" * 50)
print("  PRE-M0.2 PASS CRITERION REPORT")
print("=" * 50)

for check_name, check_value in automatic_checks.items():
    status = "PASS" if check_value else "WAIT"
    print(f"  {check_name:.<30} {status}")

print(f"  {'axes_explanation_verified':.<30} {'PASS' if AXES_EXPLANATION_VERIFIED else 'WAIT'}")
print("-" * 50)

if final_pass:
    print("  PRE-M0.2 FINAL STATUS: PASS")
else:
    print("  PRE-M0.2 FINAL STATUS: WAIT")
print("=" * 50)

---

**If STATUS is PASS**: proceed to `pre_m0_3`.

**If STATUS is WAIT**: revisit the failed gates above and resolve them before continuing.